In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.8 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you 

In [ ]:
import os
from pathlib import Path
import cv2
from ultralytics import YOLO

# 1. Configure Paths
# Path to your custom trained YOLOv8 face detection weights
YOLO_MODEL_PATH = "/kaggle/input/datasets/natalierobert/designproject/best.pt"

# Path to the dataset you just created
SOURCE_DATASET_PATH = Path("/kaggle/input/datasets/natalierobert/designproject/restructured_dataset/restructured_dataset")

# Where you want to save the new, tightly cropped face dataset
OUTPUT_DATASET_PATH = Path("/kaggle/working/")

# Valid image extensions
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def crop_faces_with_yolo():
    # 2. Load your trained YOLOv8 model
    print(f"Loading YOLOv8 face detection model from {YOLO_MODEL_PATH}...")
    model = YOLO(YOLO_MODEL_PATH)

    # Track statistics
    stats = {"processed": 0, "skipped_ds3": 0, "no_face_found": 0, "cropped_successfully": 0}

    # 3. Loop through Attentive, Distracted, Drowsy folders
    for class_folder in SOURCE_DATASET_PATH.iterdir():
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name
        target_output_dir = OUTPUT_DATASET_PATH / class_name
        target_output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\nProcessing class: {class_name}")

        for img_path in class_folder.glob("*"):
            if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            stats["processed"] += 1

            # --- RULE: Skip Dataset 3 Images ---
            # Because we prefixed them with 'ds3_' in the previous steps!
            if img_path.name.startswith("ds3_"):
                # Dataset 3 is already cropped, so we just copy it as-is to the new dataset
                import shutil
                shutil.copy2(img_path, target_output_dir / img_path.name)
                stats["skipped_ds3"] += 1
                continue

            # 4. Load the image using OpenCV
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Warning: Could not read image {img_path.name}")
                continue

            # 5. Run YOLOv8 face detection on the image
            # we use conf=0.4 (you can tweak this if it misses faces or catches false positives)
            results = model(img, conf=0.4, verbose=False)

            # Check if any face bounding boxes were detected
            if len(results) == 0 or len(results[0].boxes) == 0:
                # If YOLO fails to find a face, save the original image as a fallback
                # so you don't lose precious training data!
                cv2.imwrite(str(target_output_dir / img_path.name), img)
                stats["no_face_found"] += 1
                continue

            # Extract the first detected face (highest confidence score)
            box = results[0].boxes[0]
            # Get coordinates as integers: [xmin, ymin, xmax, ymax]
            xyxy = box.xyxy[0].cpu().numpy().astype(int)
            xmin, ymin, xmax, ymax = xyxy[0], xyxy[1], xyxy[2], xyxy[3]

            # --- OPTIONAL padding ---
            # Adding a tiny 5% boundary cushion around the face so it's not TOO tight
            h, w, _ = img.shape
            pad_w = int((xmax - xmin) * 0.05)
            pad_h = int((ymax - ymin) * 0.05)

            xmin = max(0, xmin - pad_w)
            ymin = max(0, ymin - pad_h)
            xmax = min(w, xmax + pad_w)
            ymax = min(h, ymax + pad_h)

            # 6. Crop the face out of the image array
            cropped_face = img[ymin:ymax, xmin:xmax]

            # Double check the crop isn't empty before writing
            if cropped_face.size > 0:
                output_file_path = target_output_dir / img_path.name
                cv2.imwrite(str(output_file_path), cropped_face)
                stats["cropped_successfully"] += 1
            else:
                # Fallback to original image if crop goes out of bounds weirdly
                cv2.imwrite(str(target_output_dir / img_path.name), img)
                stats["no_face_found"] += 1

    # Print a final summary report
    print("\n" + "="*40)
    print("FACE CROPPING TASK COMPLETED SUCCESSFULLY!")
    print("="*40)
    print(f"Total Images Scanned:       {stats['processed']}")
    print(f"Dataset 3 Images (Passed):  {stats['skipped_ds3']}")
    print(f"Faces Cropped via YOLOv8:  {stats['cropped_successfully']}")
    print(f"No Face Found (Kept Raw):   {stats['no_face_found']}")
    print(f"Your clean dataset is ready at: {OUTPUT_DATASET_PATH}")

if __name__ == "__main__":
    crop_faces_with_yolo()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLOv8 face detection model from /kaggle/input/datasets/natalierobert/designproject/best.pt...

Processing class: Attentive

Processing class: Drowsy

Processing class: Distracted

FACE CROPPING TASK COMPLETED SUCCESSFULLY!
Total Images Scanned:       4341
Dataset 3 Images (Passed):  1097
Faces Cropped via YOLOv8:  2897
No Face Found (Kept Raw):   347
Your clean dataset is ready at: /kaggle/working


In [ ]:
import os
import time
from IPython.display import FileLink

# 1. Ensure we are in the correct directory
os.chdir('/kaggle/working')

print("Zipping Attentive, Distracted, and Drowsy folders... Please wait...")
# Zip only our 3 target dataset folders safely
!zip -r cleaned_face_dataset.zip Attentive Distracted Drowsy > /dev/null

print("Zip archive created successfully!")

# Give the filesystem a moment to catch up
time.sleep(3)

if os.path.exists('cleaned_face_dataset.zip'):
    print(f"Success! Final file size: {os.path.getsize('cleaned_face_dataset.zip') / (1024*1024):.2f} MB")
    display(FileLink('cleaned_face_dataset.zip'))
else:
    print("Error: Zip file could not be generated.")

Zipping Attentive, Distracted, and Drowsy folders... Please wait...
Zip archive created successfully!
Success! Final file size: 165.73 MB


/kaggle/working/cleaned_face_dataset.zip